In [4]:
import sys
sys.path.insert(0, '/Users/vahid/Downloads/FERNN-master/moving_mnist_fp')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

# Import Moving MNIST dataset
from moving_mnist_dataset import MovingMNISTDataset

# Force reload of models module
import importlib
import moving_mnist_models
importlib.reload(moving_mnist_models)

print("✓ All imports successful")

✓ All imports successful


In [6]:
# Sanity Check: Test Differentiability of Modified FERNN_Cell with Optical Flow

# Generate simple synthetic Moving MNIST sequence
def generate_simple_sequence(seq_len=20, batch_size=2, height=28, width=28):
    """Generate simple moving MNIST-like sequences with constant velocity"""
    sequences = []
    
    for b in range(batch_size):
        seq = []
        # Random starting position and velocity
        x, y = np.random.randint(5, height-5, 2)
        vx, vy = np.random.uniform(-1, 1, 2)
        
        for t in range(seq_len):
            frame = np.zeros((1, height, width), dtype=np.float32)
            # Draw digit (simple circle)
            xx, yy = np.meshgrid(np.arange(width), np.arange(height))
            mask = ((xx - x)**2 + (yy - y)**2) < 9
            frame[0, mask] = 1.0
            seq.append(frame)
            
            # Update position
            x += vx
            y += vy
            
            # Wrap around
            x = x % width
            y = y % height
        
        sequences.append(np.stack(seq, axis=0))
    
    return torch.FloatTensor(np.stack(sequences, axis=0))  # (batch, seq_len, C, H, W)

# Create toy data
print("Generating synthetic Moving MNIST sequences...")
toy_seq = generate_simple_sequence(seq_len=20, batch_size=2, height=28, width=28)
print(f"Input shape: {toy_seq.shape}")

# Split into input and target
input_seq = toy_seq[:, :10]
target_seq = toy_seq[:, 10:20]
print(f"Input frames: {input_seq.shape}, Target frames: {target_seq.shape}")

# Initialize model (simple version for testing)
device = 'cpu'
hidden_channels = 32
h_kernel_size = 3
u_kernel_size = 3

# Create a simple FERNN_Cell with optical flow
try:
    from moving_mnist_models import FERNN_Cell, Seq2SeqFERNN
    
    print("\nInitializing modified FERNN_Cell with optical flow...")
    cell = FERNN_Cell(
        input_channels=1,
        hidden_channels=hidden_channels,
        h_kernel_size=h_kernel_size,
        u_kernel_size=u_kernel_size,
        v_range=2  # For optical flow estimator
    )
    cell = cell.to(device)
    
    # Forward pass - test single step
    print("\n--- Single Step Forward Pass ---")
    batch_size = input_seq.shape[0]
    h = torch.zeros(batch_size, hidden_channels, 28, 28, device=device)
    u_t_prev = input_seq[:, 0]
    u_t = input_seq[:, 1]
    
    h_next = cell(u_t, h, u_t_prev)
    print(f"✓ Forward pass successful")
    print(f"  Hidden state shape: {h_next.shape} (expected: torch.Size([{batch_size}, {hidden_channels}, 28, 28]))")
    
    # Test gradients - backprop through one step
    print("\n--- Gradient Flow Test ---")
    loss = h_next.sum()  # Dummy loss
    loss.backward()
    
    # Check if gradients exist
    grad_count = 0
    for name, param in cell.named_parameters():
        if param.grad is not None and param.grad.abs().sum() > 0:
            grad_count += 1
            print(f"✓ {name}: grad shape {param.grad.shape}, grad norm: {param.grad.norm().item():.6f}")
    
    print(f"\n✓ Gradients flowing through {grad_count} parameters")
    
    # Test full Seq2SeqFERNN
    print("\n--- Full Seq2SeqFERNN Forward/Backward ---")
    
    model = Seq2SeqFERNN(
        input_channels=1,
        hidden_channels=hidden_channels,
        height=28,
        width=28,
        output_channels=1,
        h_kernel_size=h_kernel_size,
        u_kernel_size=u_kernel_size,
        v_range=2,
        decoder_conv_layers=1
    )
    model = model.to(device)
    model.train()
    
    # Forward pass
    pred_len = 10
    output = model(input_seq.to(device), pred_len, teacher_forcing_ratio=0.0, target_seq=target_seq.to(device))
    print(f"✓ Model output shape: {output.shape} (expected: torch.Size([{batch_size}, {pred_len}, 1, 28, 28]))")
    
    # Loss and backward
    criterion = nn.MSELoss()
    loss = criterion(output, target_seq.to(device))
    print(f"✓ Loss: {loss.item():.6f}")
    
    loss.backward()
    
    # Count parameters with gradients
    total_params = sum(p.numel() for p in model.parameters())
    grad_params = sum(1 for p in model.parameters() if p.grad is not None and p.grad.abs().sum() > 0)
    total_grad_norm = sum(p.grad.norm().item()**2 for p in model.parameters() if p.grad is not None)**0.5
    
    print(f"✓ Backprop successful")
    print(f"  Total parameters: {total_params}")
    print(f"  Parameters with gradients: {grad_params}")
    print(f"  Total gradient norm: {total_grad_norm:.6f}")
    
    # Test a few optimization steps
    print("\n--- Optimization Step Test ---")
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    losses = []
    for step in range(3):
        optimizer.zero_grad()
        output = model(input_seq.to(device), pred_len, teacher_forcing_ratio=0.0, target_seq=target_seq.to(device))
        loss = criterion(output, target_seq.to(device))
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        print(f"  Step {step+1}: Loss = {loss.item():.6f}")
    
    if losses[0] > losses[-1]:
        print(f"\n✓✓✓ DIFFERENTIABILITY CHECK PASSED ✓✓✓")
        print(f"    Loss decreased from {losses[0]:.6f} to {losses[-1]:.6f}")
    else:
        print(f"\n⚠ WARNING: Loss did not decrease (may indicate gradient issues)")
        print(f"    Loss: {losses}")
    
except ImportError as e:
    print(f"Error importing models: {e}")
    print("Make sure the modified FERNN_Cell is in moving_mnist_models.py")

Generating synthetic Moving MNIST sequences...
Input shape: torch.Size([2, 20, 1, 28, 28])
Input frames: torch.Size([2, 10, 1, 28, 28]), Target frames: torch.Size([2, 10, 1, 28, 28])

Initializing modified FERNN_Cell with optical flow...

--- Single Step Forward Pass ---
✓ Forward pass successful
  Hidden state shape: torch.Size([2, 32, 28, 28]) (expected: torch.Size([2, 32, 28, 28]))

--- Gradient Flow Test ---
✓ conv_u.weight: grad shape torch.Size([32, 1, 3, 3]), grad norm: 689.488953

✓ Gradients flowing through 1 parameters

--- Full Seq2SeqFERNN Forward/Backward ---
✓ Model output shape: torch.Size([2, 10, 1, 28, 28]) (expected: torch.Size([2, 10, 1, 28, 28]))
✓ Loss: 0.035637
✓ Backprop successful
  Total parameters: 19008
  Parameters with gradients: 4
  Total gradient norm: 0.027946

--- Optimization Step Test ---
  Step 1: Loss = 0.035637
  Step 2: Loss = 0.034179
  Step 3: Loss = 0.032439

✓✓✓ DIFFERENTIABILITY CHECK PASSED ✓✓✓
    Loss decreased from 0.035637 to 0.032439
